# T/NK scVI+scANVI Reference — Debug & Write Notebook
**Purpose**: Fix write failure and clean up adata_hvg before saving final h5ad.

**What happened**: Training completed successfully. `write_h5ad` crashed because
`obs['sex']` (and possibly other columns) is `pd.StringDtype()` (nullable string),
which anndata refuses to write unless `allow_write_nullable_strings=True`.

**Approach here**:
1. Load the in-memory adata_hvg (re-run training NOT needed — just reload h5ad if already written partially, or re-run Sec 1–13 of the main script)
2. Diagnose all nullable / problematic obs columns
3. Fix dtypes
4. Clean obsm (remove stale embeddings from source object)
5. Write cleanly

**Alternative quick fix** (single line, add before write_h5ad):
```python
import anndata
anndata.settings.allow_write_nullable_strings = True
```
But this is a band-aid. Proper fix: convert to object dtype.

In [1]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import scipy.sparse as sparse
import scanpy as sc
import anndata
from pathlib import Path

sc.settings.verbosity = 1
print(f"anndata : {anndata.__version__}")
print(f"scanpy  : {sc.__version__}")

anndata : 0.11.4
scanpy  : 1.11.5


In [2]:
# ============================================================
# EDIT THESE paths if needed
# ============================================================
# If training already finished but write crashed, reload the partial h5ad
# OR just keep adata_hvg in memory from the main script and jump to Cell 5.

H5AD_IN  = "/home/h2048/data/py/0315/tnk_scarches_ref/adata_tnk_scanvi_ref_20260315_v1_2.h5ad"
H5AD_OUT = "/home/h2048/data/py/0315/tnk_scarches_ref/adata_tnk_scanvi_ref_20260315_v1_2.h5ad"
DPI = 300

# If adata_hvg is already in memory (main script ran), set LOAD_FROM_DISK = False
LOAD_FROM_DISK = True

In [3]:
if LOAD_FROM_DISK:
    print(f"Loading {Path(H5AD_IN).name} ...")
    adata_hvg = sc.read_h5ad(H5AD_IN)
    print(f"  Shape : {adata_hvg.shape}")
else:
    print("Using adata_hvg already in memory")

print(f"  .raw  : {adata_hvg.raw is not None} ({adata_hvg.raw.n_vars if adata_hvg.raw else 'N/A'} genes)")
print(f"  obsm  : {list(adata_hvg.obsm.keys())}")
print(f"  layers: {list(adata_hvg.layers.keys())}")

Loading adata_tnk_scanvi_ref_20260315_v1_2.h5ad ...
  Shape : (38904, 6000)
  .raw  : True (32723 genes)
  obsm  : []
  layers: []


In [4]:
# Diagnose ALL obs columns -- find StringDtype, object with mixed types, category issues
print("=" * 65)
print("OBS DTYPE DIAGNOSIS")
print("=" * 65)

problematic = []
for col in adata_hvg.obs.columns:
    dtype = adata_hvg.obs[col].dtype
    dtype_str = str(dtype)
    note = ""

    # Nullable string (the crash culprit)
    if isinstance(dtype, pd.StringDtype):
        note = "  <<< CRASH: StringDtype (nullable string)"
        problematic.append((col, dtype_str, 'string'))

    # Object columns with mixed types
    elif dtype == object:
        sample = adata_hvg.obs[col].dropna().head(3).tolist()
        if any(not isinstance(v, str) for v in sample):
            note = f"  <<< WARN: object with non-str ({sample[:2]})"
            problematic.append((col, dtype_str, 'mixed_object'))

    # Float64 that should maybe be float32
    elif dtype_str == 'float64':
        note = "  (float64 -- consider float32)"

    print(f"  {col:<45} {dtype_str:<20}{note}")

print(f"\nTotal problematic columns: {len(problematic)}")
for col, dtype_str, reason in problematic:
    print(f"  {col}: {dtype_str} [{reason}]")

OBS DTYPE DIAGNOSIS
  _index                                        object              
  age                                           int32               
  cDate                                         category            
  donorID                                       int32               
  nCount_RNA                                    float64               (float64 -- consider float32)
  nFeature_RNA                                  int32               
  orig.ident                                    category            
  plateID                                       category            
  sex                                           float64               (float64 -- consider float32)
  status                                        category            

Total problematic columns: 0


In [5]:
# Fix 1: Convert ALL StringDtype columns to object dtype
# This is the correct fix (not allow_write_nullable_strings)
n_fixed = 0
for col in adata_hvg.obs.columns:
    if isinstance(adata_hvg.obs[col].dtype, pd.StringDtype):
        adata_hvg.obs[col] = adata_hvg.obs[col].astype(object)
        n_fixed += 1
        print(f"  Fixed StringDtype -> object: {col}")

print(f"\nFixed {n_fixed} StringDtype columns")

# Fix 2: Re-apply category dtype to known annotation columns
# (in case StringDtype->object lost category encoding)
ANNOTATION_COLS = [
    'cell_type_L2', 'cell_type_L3', 'cell_type_L4', 'cell_type_L4_markers',
    'cell_type_L4_pipeline',
    'scanvi_label', 'scanvi_pred_L3',
    'dataset', 'sample', 'tissue', 'sex',
    'leiden_r0.5', 'leiden_r1.0',
]
for col in ANNOTATION_COLS:
    if col in adata_hvg.obs.columns:
        cur = adata_hvg.obs[col].dtype
        if cur != 'category':
            adata_hvg.obs[col] = adata_hvg.obs[col].astype('category')
            print(f"  Re-categorized: {col}  ({cur} -> category)")

print("\nDtype fix complete")


Fixed 0 StringDtype columns
  Re-categorized: sex  (float64 -> category)

Dtype fix complete


In [9]:
# Fix: '_index' is reserved by anndata h5ad writer -- rename it
if '_index' in adata_hvg.obs.columns:
    adata_hvg.obs = adata_hvg.obs.rename(columns={'_index': 'orig_index'})
    print("Renamed _index -> orig_index")

# Same check for var (less common but possible)
if '_index' in adata_hvg.var.columns:
    adata_hvg.var = adata_hvg.var.rename(columns={'_index': 'orig_index'})
    print("Renamed var._index -> orig_index")

Renamed _index -> orig_index


In [10]:
# Clean obsm: remove stale embeddings inherited from source object
# Keep only embeddings computed in THIS pipeline run
OBSM_KEEP = {'X_scvi', 'X_scanvi', 'X_umap'}
stale = [k for k in list(adata_hvg.obsm.keys()) if k not in OBSM_KEEP]

print(f"obsm before : {list(adata_hvg.obsm.keys())}")
for k in stale:
    del adata_hvg.obsm[k]
    print(f"  Removed stale obsm key: {k}")
print(f"obsm after  : {list(adata_hvg.obsm.keys())}")

obsm before : []
obsm after  : []


In [11]:
# Final check -- should show zero StringDtype entries
print("Final dtype check:")
bad = [(col, str(adata_hvg.obs[col].dtype))
       for col in adata_hvg.obs.columns
       if isinstance(adata_hvg.obs[col].dtype, pd.StringDtype)]
if bad:
    print(f"  STILL problematic ({len(bad)}): {bad}")
else:
    print("  No StringDtype columns remaining: OK")

# Check .raw still intact
print(f"\n.raw: {adata_hvg.raw is not None}")
if adata_hvg.raw is not None:
    print(f"  .raw.n_vars = {adata_hvg.raw.n_vars:,}")
    print(f"  .raw.n_obs  = {adata_hvg.raw.n_obs:,}")

# Check key obsm
for k in ['X_scvi', 'X_scanvi', 'X_umap']:
    if k in adata_hvg.obsm:
        print(f"  {k}: {adata_hvg.obsm[k].shape}")
    else:
        print(f"  MISSING: {k}")

Final dtype check:
  No StringDtype columns remaining: OK

.raw: True
  .raw.n_vars = 32,723
  .raw.n_obs  = 38,904
  MISSING: X_scvi
  MISSING: X_scanvi
  MISSING: X_umap


In [ ]:
out = Path(H5AD_OUT)
print(f"Writing {out.name} ...")
print(f"  .X shape   : {adata_hvg.shape}  (HVG, log1p)")
print(f"  .raw shape : {adata_hvg.raw.n_obs:,} x {adata_hvg.raw.n_vars:,}  (full gene, log1p)")
print(f"  .layers    : {list(adata_hvg.layers.keys())}")
print(f"  .obsm      : {list(adata_hvg.obsm.keys())}")

adata_hvg.write_h5ad(out, compression='gzip', compression_opts=9)
print(f"\nSaved: {out}")

Writing adata_tnk_scanvi_ref_20260315_v1_2.h5ad ...
  .X shape   : (38904, 6000)  (HVG, log1p)
  .raw shape : 38,904 x 32,723  (full gene, log1p)
  .layers    : []
  .obsm      : []


In [ ]:
# Quick roundtrip verify
print("Roundtrip verify ...")
tmp = sc.read_h5ad(H5AD_OUT)
print(f"  Shape         : {tmp.shape}")
print(f"  .raw.n_vars   : {tmp.raw.n_vars:,}")
print(f"  obs cols      : {len(tmp.obs.columns)}")
print(f"  obsm keys     : {list(tmp.obsm.keys())}")
print(f"  layers        : {list(tmp.layers.keys())}")
print(f"  cell_type_L3  : {tmp.obs['cell_type_L3'].value_counts().to_dict()}")
del tmp
print("\nRoundtrip OK")

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(21, 6))

sc.pl.embedding(adata_hvg, basis='umap', color='cell_type_L3',
                title='L3', ax=axes[0], show=False,
                legend_loc='right margin', legend_fontsize=6)
sc.pl.embedding(adata_hvg, basis='umap', color='cell_type_L2',
                title='L2', ax=axes[1], show=False,
                legend_loc='on data', legend_fontsize=8)
sc.pl.embedding(adata_hvg, basis='umap', color='scanvi_pred_prob',
                title='scANVI confidence', ax=axes[2], show=False,
                color_map='RdYlGn', vmin=0, vmax=1)

for ax in fig.axes:
    for coll in ax.collections:
        coll.set_rasterized(True)

plt.tight_layout()
fig_path = Path(H5AD_OUT).parent / 'figures' / 'debug_umap_check.pdf'
fig_path.parent.mkdir(exist_ok=True)
fig.savefig(fig_path, bbox_inches='tight', dpi=DPI)
plt.show()
print(f"Saved: {fig_path.name}")